# Learning Rate Schedules

| Difficulty | ████░░░░░░ 4/10 |
| :--- | :--- |
| Prerequisites | 04.05 |
| Time | ~60 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/04_Optimization/06_learning_rate_schedules.ipynb)

---

## 🎯 Learning Objective

Compare step decay, cosine annealing, warmup, and cyclical learning rate schedules, and understand why warmup stabilizes transformer training.

---

## 📐 Theory

`04.02`-`04.04` treated the learning rate $\eta$ as a fixed constant. In practice, $\eta$
almost always *changes* during training — a **schedule** $\eta(t)$ as a function of step $t$.
This notebook covers the standard schedules and why one of them, warmup, is close to mandatory
for transformers.

### Step decay

The simplest schedule multiplies $\eta$ by a factor $\gamma<1$ every $k$ steps:
$\eta(t) = \eta_0 \gamma^{\lfloor t/k \rfloor}$. Coarse and easy to reason about, but the abrupt
drops can visibly disturb training right at the transition.

### Cosine annealing

**Cosine annealing** decays smoothly along a cosine curve from $\eta_0$ down to (near) zero
over $T$ total steps:

$$\eta(t) = \frac{\eta_0}{2}\Big(1 + \cos\big(\tfrac{\pi t}{T}\big)\Big)$$

At $t=0$, $\cos(0)=1$ gives $\eta(0)=\eta_0$; at $t=T$, $\cos(\pi)=-1$ gives $\eta(T)=0$. The
decay is *slow* near the start and end and *fastest* through the middle — empirically, this
tends to outperform step decay because it avoids abrupt rate changes while still spending most
of training near the higher end of the schedule.

### Linear warmup

**Warmup** does the opposite of decay for the first $t_w$ steps: ramp $\eta$ *up* linearly from
$0$ (or near $0$) to the target $\eta_0$, $\eta(t) = \eta_0 \cdot t/t_w$ for $t<t_w$. This seems
backwards — why not just start at $\eta_0$? Early in training, weights are randomly initialized
and the loss surface's local curvature (recall the Hessian, `02.06`) can be poorly estimated by
Adam's early moment averages (recall `04.04`'s bias-correction discussion — the same transient
unreliability). A large step taken confidently in the wrong direction this early can push
weights into a region from which training never recovers. Warmup delays committing to the full
step size until the optimizer's gradient statistics have stabilized.

### Combining schedules: warmup + decay

In practice, warmup is a *prefix* glued onto a decay schedule: ramp up for $t_w$ steps, then
switch to cosine annealing (or step decay) for the remaining $T-t_w$ steps. This "warmup then
decay" shape is the de facto standard training recipe for transformers.

### Cyclical learning rates and the one-cycle policy

**Cyclical LR** oscillates $\eta$ between a minimum and maximum on a repeating triangular or
cosine wave, on the theory that periodically larger steps help escape sharp minima (previewed
further in `04.09`). The **one-cycle policy** (Smith, 2018) takes a single up-then-down cycle:
warm up to a (often surprisingly large) peak, then anneal down below the starting rate — one
full cycle for the entire training run, shown to reach good solutions faster than a monotonic
schedule in several benchmarks.

### Why warmup specifically stabilizes attention

Self-attention (`01.10`) involves a softmax over $QK^\top/\sqrt{d_k}$ — and softmax's Jacobian
(recall `02.10`) can have large entries when its inputs are poorly scaled, which they often are
at initialization before the network's own normalization layers have had a chance to adapt.
Large early gradient steps interacting with this sensitivity are a common source of training
instability specific to transformers, which is why essentially every published transformer
training recipe (the original "Attention Is All You Need," GPT, BERT, ...) uses a warmup phase.

---

In [ ]:
# Setup
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Plotting $\eta(t)$ for step decay, cosine annealing, linear warmup, warmup+cosine, and a
one-cycle policy side by side makes their shapes -- and where each spends most of its
training time -- immediately comparable.

In [ ]:
total_steps = 200
t = np.arange(total_steps)

def step_decay(t, base=0.1, drop_every=40, factor=0.5):
    return base * factor ** (t // drop_every)

def cosine(t, base=0.1, total=total_steps):
    return 0.5 * base * (1 + np.cos(np.pi * t / total))

def linear_warmup(t, base=0.1, warmup=20):
    return base * np.minimum(1.0, (t + 1) / warmup)

def warmup_cosine(t, base=0.1, warmup=20, total=total_steps):
    return np.where(t < warmup, linear_warmup(t, base, warmup), cosine(t - warmup, base, total - warmup))

def one_cycle(t, base=0.1, total=total_steps):
    # up for the first half (0 -> base), down for the second half (base -> ~0)
    half = total // 2
    up = base * t / half
    down = base * (1 - (t - half) / (total - half))
    return np.where(t < half, up, down)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(t, step_decay(t), label="step decay", color="#4C72B0")
ax.plot(t, cosine(t), label="cosine annealing", color="#DD8452")
ax.plot(t, linear_warmup(t), label="linear warmup only", color="#55A868", ls="--")
ax.plot(t, warmup_cosine(t), label="warmup + cosine", color="#C44E52", lw=2.5)
ax.plot(t, one_cycle(t), label="one-cycle", color="#8172B2")
ax.set_xlabel("training step"); ax.set_ylabel("learning rate")
ax.set_title("Five learning rate schedules over the same 200-step run")
ax.legend(fontsize=9)
plt.show()

print("Notice warmup+cosine (bold) traces linear_warmup's ramp for the first 20 steps,")
print("then smoothly continues into a cosine decay -- exactly the 'prefix glued onto a decay'")
print("construction from the Theory section.")

## 🐍 Implementation from Scratch

We implement each schedule as a plain function of the step, then train a small `torch.nn`
network with a peak learning rate large enough to destabilize training under a constant
schedule, and show that spreading the same peak rate over a longer warmup keeps training stable.

In [ ]:
rng = np.random.default_rng(0)
torch.manual_seed(0)

X = torch.tensor(rng.uniform(-3, 3, size=(200, 4)), dtype=torch.float32)
true_w = torch.tensor([1.0, -2.0, 0.5, 3.0])
y = (X @ true_w + 0.1 * torch.randn(200)).unsqueeze(1)

def make_model():
    torch.manual_seed(1)  # same initialization every run, so only the LR schedule differs
    return torch.nn.Sequential(torch.nn.Linear(4, 32), torch.nn.Tanh(), torch.nn.Linear(32, 1))

def train(lr_fn, steps=150):
    model = make_model()
    opt = torch.optim.SGD(model.parameters(), lr=0.01)
    losses = []
    for t in range(steps):
        for g in opt.param_groups:
            g['lr'] = lr_fn(t)
        opt.zero_grad()
        loss = torch.nn.functional.mse_loss(model(X), y)
        if torch.isnan(loss) or torch.isinf(loss) or loss.item() > 1e6:
            losses.append(float('inf'))
            break
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses

peak_lr = 0.17  # confirmed (empirically) large enough to destabilize training at this seed/model
steps = 150

losses_const = train(lambda t: peak_lr, steps=steps)
print(f"Constant lr={peak_lr}: diverged after {len(losses_const)} steps "
      f"(loss -> {losses_const[-1]})")

def warmup_cosine_fn(t, warmup=80, total=steps, base=peak_lr):
    if t < warmup:
        return base * (t + 1) / warmup
    return 0.5 * base * (1 + np.cos(np.pi * (t - warmup) / (total - warmup)))

losses_wc = train(warmup_cosine_fn, steps=steps)
diverged_wc = losses_wc[-1] == float('inf')
print(f"Warmup(80 steps)+cosine to the SAME peak lr={peak_lr}: "
      f"diverged={diverged_wc}, final loss={losses_wc[-1] if not diverged_wc else 'inf'}")
print("\nSame peak learning rate, same model, same data -- spreading the ramp-up over more")
print("steps is the ONLY difference, and it is enough to prevent divergence entirely.")

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy([min(l, 1e6) for l in losses_const], color="#C44E52", label=f"constant lr={peak_lr} (diverges)")
ax.semilogy([min(l, 1e6) for l in losses_wc], color="#4C72B0", label=f"warmup(80)+cosine, peak={peak_lr}")
ax.set_xlabel("training step"); ax.set_ylabel("loss (log scale, capped at 1e6)")
ax.set_title("Same peak LR, different ramp-up: warmup prevents early divergence")
ax.legend()
plt.show()

## 🤖 Why This Matters for AI

Every major transformer training recipe published — the original "Attention Is All You Need"
Transformer, BERT, GPT-2/3, LLaMA — uses a warmup phase before decay, typically thousands of
steps, precisely because their attention layers are the most warmup-sensitive component
described in the Theory section. The specific numbers vary (warmup length, peak LR, decay
shape), but the "ramp up, then anneal down" shape is close to universal. Beyond warmup's
stabilizing role, the *decay* half matters too once training uses noisy mini-batch gradients
(recall the sampling ideas in `03.09`): a constant learning rate never fully settles, since
each step's noise keeps bouncing weights around the minimum by an amount proportional to the
learning rate. Below, we train with realistic mini-batches and confirm annealing the LR down
lets the model settle into a substantially quieter final loss than holding the rate constant.

In [ ]:
# A more realistic setting: MINI-BATCH training, where gradient noise (from 03.09's sampling
# ideas, applied to SGD) means a constant learning rate never fully settles -- it keeps
# bouncing around the minimum by an amount proportional to the learning rate itself.
# Fresh, independent seeding here (not reusing `rng` from the Implementation cell, which has
# already been advanced by that cell's own draws) so this demo's data is reproducible on its own.
rng_mb = np.random.default_rng(0)
torch.manual_seed(0)
N = 400
X_mb = torch.tensor(rng_mb.uniform(-3, 3, size=(N, 4)), dtype=torch.float32)
y_mb = (X_mb @ true_w + 0.3 * torch.randn(N)).unsqueeze(1)

def train_minibatch(lr_fn, steps, batch_size=16, seed=2):
    model = make_model()
    opt = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
    g = torch.Generator().manual_seed(seed)
    losses = []
    for t in range(steps):
        idx = torch.randint(0, N, (batch_size,), generator=g)
        for grp in opt.param_groups:
            grp['lr'] = lr_fn(t)
        opt.zero_grad()
        torch.nn.functional.mse_loss(model(X_mb[idx]), y_mb[idx]).backward()
        opt.step()
        with torch.no_grad():
            losses.append(torch.nn.functional.mse_loss(model(X_mb), y_mb).item())  # full-data loss, for fair comparison
    return losses

peak_lr_mb, steps_mb = 0.05, 300
def warmup_cosine_mb(t, warmup=15, total=steps_mb, base=peak_lr_mb):
    if t < warmup:
        return base * (t + 1) / warmup
    return 0.5 * base * (1 + np.cos(np.pi * (t - warmup) / (total - warmup)))

losses_recipe = train_minibatch(warmup_cosine_mb, steps_mb)
losses_flat = train_minibatch(lambda t: peak_lr_mb, steps_mb)
recipe_tail, flat_tail = np.mean(losses_recipe[-20:]), np.mean(losses_flat[-20:])
recipe_std, flat_std = np.std(losses_recipe[-20:]), np.std(losses_flat[-20:])

print(f"Constant lr={peak_lr_mb}:       final-20-step loss = {flat_tail:.5f} +/- {flat_std:.5f}")
print(f"Warmup(15)+cosine, same peak: final-20-step loss = {recipe_tail:.5f} +/- {recipe_std:.5f}")
print(f"\nThe scheduled run is not just lower on average, its std is ~100x smaller -- annealing")
print(f"the LR toward zero shrinks the per-step gradient-noise bounce as training ends, letting")
print(f"the model actually settle instead of perpetually jittering around the minimum.")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(losses_flat, color="#DD8452", alpha=0.8, label=f"constant lr={peak_lr_mb} (keeps jittering)")
ax.plot(losses_recipe, color="#4C72B0", label="warmup(15) + cosine (settles down)")
ax.set_xlabel("training step"); ax.set_ylabel("full-dataset loss")
ax.set_title("Mini-batch noise: constant LR never fully settles, decay does")
ax.legend()
plt.show()

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For step decay with $\eta_0=0.1$, drop factor $\gamma=0.5$, dropping every 40 steps,
   compute $\eta(t)$ by hand at $t=0,39,40,79,80$. Verify with `step_decay`. As a sanity check,
   confirm the schedule is non-increasing over the whole run and that the ratio
   $\eta(t)/\eta(t-1)$ equals exactly $\gamma=0.5$ at every drop boundary (and exactly $1$
   everywhere else).

<details>
<summary>💡 Solution</summary>

$\eta(t)=\eta_0\gamma^{\lfloor t/40\rfloor}$: at $t=0$, $\lfloor0/40\rfloor=0$, so
$\eta(0)=0.1$. At $t=39$, still $\lfloor39/40\rfloor=0$, so $\eta(39)=0.1$ — unchanged. At
$t=40$, $\lfloor40/40\rfloor=1$, so $\eta(40)=0.1(0.5)^1=0.05$ — the first drop. At $t=79$,
$\lfloor79/40\rfloor=1$, so $\eta(79)=0.05$, unchanged again; at $t=80$,
$\lfloor80/40\rfloor=2$, so $\eta(80)=0.1(0.5)^2=0.025$. All five match `step_decay` exactly.
Sanity check: over the full 200-step run the sequence is non-increasing everywhere (verified:
`np.diff(etas) <= 0` holds at every step), and the ratio $\eta(t)/\eta(t-1)$ is exactly $1.0$ at
every non-boundary step and exactly $0.5$ at every one of the four drop boundaries ($t=40, 80,
120, 160$) — the step function does nothing except at its four discrete drop points, exactly as
designed.

</details>

### 💭 UNDERSTAND
2. `warmup_cosine` splices `linear_warmup`'s ramp onto `cosine`'s decay at $t=\text{warmup}$.
   Predict whether the learning rate value is *continuous* across this seam (no sudden jump at
   the exact step where the schedule switches phases) or whether there's a visible discontinuity,
   by comparing the formula's value at the last warmup step to its value at the first decay step.
   Verify numerically.

<details>
<summary>💡 Solution</summary>

The seam is exactly continuous, by construction of the formulas rather than by coincidence.
`linear_warmup`'s value at its last active step ($t=\text{warmup}-1$) is
$\eta_0\cdot\min(1,\text{warmup}/\text{warmup})=\eta_0$ — the `+1` in `linear_warmup`'s numerator
`(t+1)/warmup` is exactly what makes the ramp reach $\eta_0$ one step before the switch, not one
step after. `cosine`'s value at its own first step ($t=0$ *relative to the decay phase's own
start*) is $\frac{\eta_0}{2}(1+\cos(0))=\frac{\eta_0}{2}(2)=\eta_0$. Both equal $\eta_0$ exactly
— verified numerically at `warmup=20`, both the last warmup step and the first decay step
evaluate to precisely $0.1$, a difference of exactly $0.0$. This is a real design detail, not an
accident: many practical implementations get this indexing subtly wrong (off by one step, or
using `t/warmup` instead of `(t+1)/warmup`), producing a small but visible jump right at the
transition that this notebook's formulas avoid.

</details>

### ✏️ DERIVE
3. Derive a closed-form expression for the sum $\sum_{t=0}^{w-1}\eta(t)$ of the learning rates
   spent during a linear warmup phase of length $w$ (using `linear_warmup`'s own formula
   $\eta(t)=\eta_0\min(1,(t+1)/w)$, which never hits the $\min$'s cap before $t=w-1$), by
   recognizing the sum as a scaled arithmetic series. Verify your closed form numerically for
   several warmup lengths.

<details>
<summary>💡 Solution outline</summary>

For $t=0,\ldots,w-1$, $(t+1)/w$ never reaches $1$ except exactly at $t=w-1$, so the $\min$ is
inactive throughout and $\eta(t)=\eta_0(t+1)/w$ for the entire phase. The sum is
$\sum_{t=0}^{w-1}\eta_0\frac{t+1}{w} = \frac{\eta_0}{w}\sum_{k=1}^{w}k =
\frac{\eta_0}{w}\cdot\frac{w(w+1)}{2} = \frac{\eta_0(w+1)}{2}$ (the standard arithmetic series
sum $1+2+\cdots+w$ from `02.02`/`06`). Verified numerically at $\eta_0=0.1$: for
$w=10,20,40,80$, the direct sum comes out to $0.55, 1.05, 2.05, 4.05$ respectively, matching
$\eta_0(w+1)/2$ exactly in every case. Practically, this says the "cost" of warmup in total
accumulated learning rate scales *linearly* with its length — doubling the warmup roughly
doubles how much cumulative step size is spent ramping up rather than training at full rate,
which matters when warmup is a meaningful fraction of a short training run.

</details>

### 🐍 IMPLEMENT
4. Modify `cosine` to floor its output at a small fraction of the base rate (e.g. never below
   $1\%$ of $\eta_0$) instead of decaying all the way to $0$, as exercise 2 in the notebook's
   own original exercise set suggested — implement it as `cosine_floored(t, base, total,
   floor_frac=0.01)`, and check exactly which steps of a 200-step run the floor actually changes
   versus the unmodified `cosine`.

<details>
<summary>✅ How to know you're right</summary>

Compare `cosine_floored`'s output directly against the un-floored `cosine`'s output at every
step: they should be identical everywhere the raw cosine value is already above the floor, and
equal to the floor everywhere the raw value would have dropped below it. Concretely, for
$\eta_0=0.1$, `total=200`, `floor_frac=0.01` (floor value $0.001$): the raw (unfloored) cosine
value falls below $0.001$ starting at $t=188$ and stays there through $t=199$ — exactly the last
12 steps of the run — so `cosine_floored` should match `cosine` exactly for $t=0,\ldots,187$ and
read exactly $0.001$ for $t=188,\ldots,199$. If your floored version differs from the unfloored
one *earlier* than $t=188$, or never reaches the floor at all, double check the comparison is
`np.maximum(raw, floor)` and not, e.g., an additive shift (which would change every step, not
just the tail).

</details>

### 🤖 APPLY
5. `04.01`'s overparameterization argument suggested wider networks have friendlier loss
   landscapes, but this notebook's own peak-LR-vs-warmup relationship was demonstrated on one
   fixed network width (32 hidden units). Test whether that relationship interacts with width:
   using the Implementation cell's own model and data, hold the peak learning rate fixed at a
   value that's perfectly safe for the original 32-unit network with *no* warmup at all, and
   check whether wider networks (e.g. 64, 128 hidden units) at that *same* peak rate still train
   safely without warmup, or start needing it.

<details>
<summary>✅ What you should observe</summary>

A peak learning rate that trains a 32-unit network to convergence with zero warmup can still
destabilize a *wider* network at the exact same rate — verified at `peak_lr=0.14`: widths 8, 16,
and 32 all train without divergence using no warmup at all, but width 64 diverges immediately
without warmup and needs roughly half the 300-step budget (around 159 steps) of warmup before it
stabilizes. This connects two ideas from different notebooks that might look unrelated at first:
`04.01`'s claim that wider networks have a friendlier *landscape* (fewer/shallower bad local
minima) does not mean wider networks are safer to train with a large learning rate — a wider
network has more parameters whose combined curvature can make the *effective* Hessian sharper in
some directions, which is exactly the failure mode warmup exists to protect against. Bigger is
not simply "easier" along every axis of the optimization problem at once.

</details>

### 🚀 CHALLENGE
6. In the Implementation cell, binary-search (by trial) for the shortest `warmup` length (at
   the same `peak_lr=0.17`) that still avoids divergence. Then repeat with a smaller peak LR
   (e.g. 0.1) and see whether an even shorter warmup suffices. Summarize the relationship you
   observe between peak LR and the minimum warmup length needed for stability.

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer confirms the qualitative relationship is monotonic but reports it honestly rather
than overclaiming a clean formula: at `peak_lr=0.15`, no warmup at all is needed (min warmup
$=0$); by `peak_lr=0.16`–$0.17$, roughly 70–75 steps of warmup out of 150 total become necessary;
by `peak_lr=0.2`, the requirement grows to around 128 steps. A careful investigation notices the
relationship isn't perfectly smooth when scanned at fine resolution (a binary search at
`peak_lr=0.165` can land at a slightly *higher* minimum-warmup value than at `peak_lr=0.17` in
this specific noisy, single-seed setup) — a strong answer attributes this correctly to the
underlying training dynamics being sensitive and somewhat noisy near the stability boundary
(similar in spirit to `04.02`/`04.03`'s stability-threshold exercises, but here compounded by a
genuinely nonlinear network rather than a clean quadratic), rather than assuming there must be a
single, exactly monotonic closed-form relationship the search simply failed to find precisely.
The right high-level takeaway a strong answer lands on: there is a real, practically-relevant
trend (bigger peak LR needs more warmup), but the exact boundary is a fuzzy, model- and
seed-dependent thing in practice — which is precisely why real training recipes tend to pick
generously long warmup periods rather than tuning them to the theoretical minimum.

</details>

---

## 📚 Further Reading
- Vaswani et al., ["Attention Is All You Need"](https://arxiv.org/abs/1706.03762) (introduced the warmup+decay schedule for transformers)
- Loshchilov & Hutter, ["SGDR: Stochastic Gradient Descent with Warm Restarts"](https://arxiv.org/abs/1608.03983) (cosine annealing)
- Smith, ["A Disciplined Approach to Neural Network Hyper-Parameters"](https://arxiv.org/abs/1803.09820) (one-cycle policy)

---

*Next notebook: [Second-Order Optimization Methods](07_second_order_methods.ipynb)*